# DNA Shape & Dinucleotide Feature Analysis

Executes the staged analysis from `DNA_SHAPE_ANALYSIS_PLAN.md` for RET nucleosomes and GC-matched controls. Outputs are written to `output/dna_shape_analysis/`.

In [1]:
from pathlib import Path
import sys
import yaml

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'src/markov_execution/markov_sweep.yaml').exists():
            return path
    raise RuntimeError('Could not find project root containing src/markov_execution/markov_sweep.yaml')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from analysis import dna_shape_features as dsf

CONFIG_PATH = PROJECT_ROOT / 'src/markov_execution/markov_sweep.yaml'
OUTPUT_DIR = PROJECT_ROOT / 'output/dna_shape_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open() as handle:
    config = yaml.safe_load(handle)

def resolve_config_path(value):
    text = str(value)
    if text.startswith('hamnucret_fasta_dir/'):
        return Path(config['hamnucret_fasta_dir']) / text.split('/', 1)[1]
    path = Path(text)
    return path if path.is_absolute() else PROJECT_ROOT / path

SPRM_ROOT = resolve_config_path(config['sprm_root'])
DATASET_LABELS = {
    'ret_all_stable147_refined': 'RET',
    'ctrl02_random_genome_gcmatched_stable147_refined': 'ctrl02',
    'ctrl03_som_gcmatched_stable147_refined': 'ctrl03',
}
DATASET_FASTA = {
    dataset: resolve_config_path(config['dataset_fasta'][dataset]['path'])
    for dataset in DATASET_LABELS
}

PROJECT_ROOT, SPRM_ROOT, OUTPUT_DIR

(PosixPath('/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis'),
 PosixPath('/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/SPRM_data'),
 PosixPath('/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis'))

## Load FASTA and SPRM Energies

`dF_total` is taken from the fully wrapped state, `left_open=0` and `right_open=0`, for the 2D scatter requested in the plan.

In [2]:
datasets = {}
all_rows = []

for dataset, label in DATASET_LABELS.items():
    rows = dsf.load_dataset(dataset, DATASET_FASTA[dataset], SPRM_ROOT, left_open=0, right_open=0)
    warnings = dsf.validate_sequences({row['seq_id']: row['sequence'] for row in rows})
    if warnings:
        print(dataset, warnings)
    datasets[label] = rows
    all_rows.extend(rows)
    print(f'{label:6s} {len(rows):6d} sequences  FASTA={DATASET_FASTA[dataset]}')

dsf.add_dinucleotide_features(all_rows)
print(f'Loaded {len(all_rows)} total sequences')

RET     27778 sequences  FASTA=/home/pol_schiessel/maya620d/pol/Projects/Codebase/HAMNucRetSeq_pipeline/RET_output/all/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa
ctrl02  15594 sequences  FASTA=/home/pol_schiessel/maya620d/pol/Projects/Codebase/HAMNucRetSeq_pipeline/SOM_output/control_sets/ctrl02_random_genome_gcmatched/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa
ctrl03  19944 sequences  FASTA=/home/pol_schiessel/maya620d/pol/Projects/Codebase/HAMNucRetSeq_pipeline/SOM_output/control_sets/ctrl03_som_gcmatched/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa
Loaded 63316 total sequences


## Stage 1: Dinucleotide Composition

In [3]:
comparison_rows = []
for ctrl_label in ['ctrl02', 'ctrl03']:
    comparison_rows.extend(dsf.compare_dinucleotides(datasets['RET'], datasets[ctrl_label], ctrl_label))

dsf.write_table(comparison_rows, OUTPUT_DIR / 'dinucleotide_comparison.csv')

try:
    import pandas as pd
    display(pd.DataFrame(comparison_rows).sort_values(['comparison', 'p_value']).head(32))
except ImportError:
    for row in comparison_rows[:12]:
        print(row)

dsf.plot_dinucleotide_barplot(comparison_rows, OUTPUT_DIR / 'dinucleotide_barplot.png')
print(OUTPUT_DIR / 'dinucleotide_comparison.csv')
print(OUTPUT_DIR / 'dinucleotide_barplot.png')

,comparison,feature,ret_mean,ret_std,ctrl_mean,ctrl_std,diff,p_value,effect_size,n_ret,n_ctrl
0,RET_vs_ctrl02,CG,0.092610,0.045508,0.069587,0.045035,0.023022,0.000000e+00,0.508530,27778,15594
1,RET_vs_ctrl02,GC,0.128605,0.036508,0.114014,0.035647,0.014591,0.000000e+00,0.404401,27778,15594
2,RET_vs_ctrl02,TG,0.058244,0.025796,0.067688,0.029743,-0.009444,3.197813e-255,-0.339214,27778,15594
3,RET_vs_ctrl02,CA,0.058140,0.025772,0.067580,0.029686,-0.009440,5.726783e-251,-0.339600,27778,15594
4,RET_vs_ctrl02,GT,0.037883,0.018514,0.042361,0.022884,-0.004478,5.025545e-79,-0.215137,27778,15594
5,RET_vs_ctrl02,AC,0.037944,0.018749,0.042036,0.022825,-0.004092,3.559547e-66,-0.195918,27778,15594
8,RET_vs_ctrl02,AT,0.013851,0.011990,0.015830,0.012950,-0.001979,5.795498e-56,-0.158598,27778,15594
6,RET_vs_ctrl02,AG,0.068892,0.026552,0.072568,0.030054,-0.003676,1.127614e-36,-0.129632,27778,15594
7,RET_vs_ctrl02,CT,0.068830,0.026597,0.071758,0.029988,-0.002928,8.457172e-27,-0.103304,27778,15594
11,RET_vs_ctrl02,CC,0.139740,0.051688,0.139030,0.064284,0.000711,1.394839e-11,0.012185,27778,15594


/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/dinucleotide_comparison.csv
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/dinucleotide_barplot.png


In [4]:
ALPHA_BONFERRONI = 0.003
significant_dn = dsf.significant_features(comparison_rows, alpha=ALPHA_BONFERRONI)
print('Bonferroni-significant dinucleotides:', significant_dn)

if significant_dn:
    dsf.plot_positional_dinucleotides(datasets, significant_dn, OUTPUT_DIR / 'positional_dinuc_profiles.png')
    print(OUTPUT_DIR / 'positional_dinuc_profiles.png')
else:
    print('No positional dinucleotide plot written because no dinucleotide passed alpha=0.003.')

Bonferroni-significant dinucleotides: ['AA', 'AC', 'AG', 'AT', 'CA', 'CC', 'CG', 'CT', 'GC', 'GG', 'GT', 'TA', 'TC', 'TG', 'TT']
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/positional_dinuc_profiles.png


## Stage 2: DNA Shape Features

If Rohs-lab pentamer tables are available, set the paths below. Expected format is two columns: pentamer and value. If paths are left as `None`, the notebook still computes the approximate dinucleotide-level MGW fallback from the plan.

In [5]:
SHAPE_TABLE_PATHS = {
    'MGW': None,
    'ProT': None,
    'Roll': None,
    'HelT': None,
}
SHAPE_FEATURE_TYPES = {
    'MGW': 'nucleotide',
    'ProT': 'nucleotide',
    'Roll': 'step',
    'HelT': 'step',
}

shape_tables = {}
for name, table_path in SHAPE_TABLE_PATHS.items():
    if table_path is None:
        continue
    path = Path(table_path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    shape_tables[name] = (dsf.load_shape_table(path), SHAPE_FEATURE_TYPES[name])
    print(f'Loaded {name}: {len(shape_tables[name][0])} pentamers from {path}')

shape_features = dsf.add_shape_features(all_rows, shape_tables=shape_tables, include_simple_mgw=True)
print('Shape features available:', shape_features)

Shape features available: ['MGW_simple']


In [6]:
shape_comparisons = []
for feature in shape_features:
    for ctrl_label in ['ctrl02', 'ctrl03']:
        shape_comparisons.append(dsf.compare_shape_feature(datasets['RET'], datasets[ctrl_label], ctrl_label, feature))

dsf.write_table(shape_comparisons, OUTPUT_DIR / 'shape_comparison.csv')
dsf.plot_shape_distributions(datasets, shape_features, OUTPUT_DIR / 'shape_distributions.png')
dsf.plot_shape_positional_profiles(datasets, shape_features, OUTPUT_DIR / 'shape_positional_profiles.png')

try:
    import pandas as pd
    display(pd.DataFrame(shape_comparisons).sort_values('p_value'))
except ImportError:
    for row in shape_comparisons:
        print(row)

print(OUTPUT_DIR / 'shape_comparison.csv')
print(OUTPUT_DIR / 'shape_distributions.png')
print(OUTPUT_DIR / 'shape_positional_profiles.png')

,comparison,feature,ret_mean,ret_std,ctrl_mean,ctrl_std,diff,p_value,effect_size,n_ret,n_ctrl
0,RET_vs_ctrl02,MGW_simple,4.593371,0.077532,4.568338,0.073760,0.025033,1.292998e-267,0.330818,27778,15594
1,RET_vs_ctrl03,MGW_simple,4.593371,0.077532,4.590002,0.073637,0.003369,1.121597e-05,0.044561,27778,19944


/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/shape_comparison.csv
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/shape_distributions.png
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/shape_positional_profiles.png


## Stage 3: 2D Energy Scatter

The scatter uses the strongest significant dinucleotide if present; otherwise it falls back to the strongest absolute dinucleotide difference. A shape scatter is also written for the first available shape feature.

In [7]:
if significant_dn:
    scatter_dn = significant_dn[0]
else:
    scatter_dn = max(comparison_rows, key=lambda row: abs(float(row['diff'])))['feature']

dn_column = f'dn_{scatter_dn}'
dsf.plot_energy_feature_scatter(datasets, dn_column, OUTPUT_DIR / '2d_scatter_dinucleotide.png')
print(f'Dinucleotide scatter feature: {dn_column}')
print(OUTPUT_DIR / '2d_scatter_dinucleotide.png')

if shape_features:
    shape_column = f'shape_{shape_features[0]}_mean'
    dsf.plot_energy_feature_scatter(datasets, shape_column, OUTPUT_DIR / '2d_scatter_shape.png')
    print(f'Shape scatter feature: {shape_column}')
    print(OUTPUT_DIR / '2d_scatter_shape.png')

Dinucleotide scatter feature: dn_AA
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/2d_scatter_dinucleotide.png
Shape scatter feature: shape_MGW_simple_mean
/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis/2d_scatter_shape.png


## Stage 4: Summary

In [8]:
summary_lines = []
summary_lines.append('DNA shape and dinucleotide analysis summary')
summary_lines.append(f'Output directory: {OUTPUT_DIR}')
summary_lines.append('')

for comparison in sorted({row['comparison'] for row in comparison_rows}):
    rows = [row for row in comparison_rows if row['comparison'] == comparison]
    rows = sorted(rows, key=lambda row: abs(float(row['diff'])), reverse=True)
    summary_lines.append(f'=== {comparison}: top dinucleotides by absolute mean difference ===')
    summary_lines.append(f"{'Step':<6} {'RET mean':>10} {'CTRL mean':>10} {'Diff':>10} {'p-value':>12} {'Effect':>10}")
    for row in rows[:8]:
        mark = ' *' if float(row['p_value']) < ALPHA_BONFERRONI else ''
        summary_lines.append(
            f"{row['feature']:<6} {float(row['ret_mean']):>10.4f} {float(row['ctrl_mean']):>10.4f} "
            f"{float(row['diff']):>+10.4f} {float(row['p_value']):>12.3e} {float(row['effect_size']):>+10.3f}{mark}"
        )
    summary_lines.append('')

if shape_comparisons:
    summary_lines.append('=== Shape features ===')
    for row in sorted(shape_comparisons, key=lambda row: float(row['p_value'])):
        summary_lines.append(
            f"{row['comparison']} {row['feature']}: RET={float(row['ret_mean']):.4f}, "
            f"CTRL={float(row['ctrl_mean']):.4f}, diff={float(row['diff']):+.4f}, "
            f"p={float(row['p_value']):.3e}, effect={float(row['effect_size']):+.3f}"
        )

summary_text = '\n'.join(summary_lines)
(OUTPUT_DIR / 'analysis_summary.txt').write_text(summary_text)
print(summary_text)
print('\nWrote', OUTPUT_DIR / 'analysis_summary.txt')

DNA shape and dinucleotide analysis summary
Output directory: /home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis/output/dna_shape_analysis

=== RET_vs_ctrl02: top dinucleotides by absolute mean difference ===
Step     RET mean  CTRL mean       Diff      p-value     Effect
CG         0.0926     0.0696    +0.0230    0.000e+00     +0.509 *
GC         0.1286     0.1140    +0.0146    0.000e+00     +0.404 *
TG         0.0582     0.0677    -0.0094   3.198e-255     -0.339 *
CA         0.0581     0.0676    -0.0094   5.727e-251     -0.340 *
GT         0.0379     0.0424    -0.0045    5.026e-79     -0.215 *
AC         0.0379     0.0420    -0.0041    3.560e-66     -0.196 *
AG         0.0689     0.0726    -0.0037    1.128e-36     -0.130 *
CT         0.0688     0.0718    -0.0029    8.457e-27     -0.103 *

=== RET_vs_ctrl03: top dinucleotides by absolute mean difference ===
Step     RET mean  CTRL mean       Diff      p-value     Effect
CG         0.0926     0.0791    +0.0135   5.937e-2